Training a Transformer from scratch is the ultimate rite of passage in modern Deep Learning. Since you are already comfortable with the PyTorch mechanics of the attention block, we are going to build and train a Generative Causal Language Model (a mini-GPT).

To do this, we are shifting from the Encoder architecture (which looks at the whole sequence at once) to a Decoder architecture. The critical difference is the Causal Mask—a mathematical trick that prevents the network from looking into the future and "cheating" while trying to predict the next token.

The Data Pipeline (Character-Level)

Before we train, we need data. We will use a simple text string, break it into characters, create a vocabulary, and build an input-target pipeline where the "target" is just the "input" shifted by one position into the future.

In [5]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# 1. The Dataset
text = "deep learning is fascinating and transformers are powerful."
chars = sorted(list(set(text)))
vocab_size = len(chars)

# 2. Tokenizers (Character to Integer mappings)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# 3. Create Training Data
data = torch.tensor(encode(text), dtype=torch.long)
seq_length = 8

def get_batch():
    # Randomly select starting indices for our sequence
    ix = torch.randint(len(data) - seq_length, (4,)) # Batch size of 4
    x = torch.stack([data[i:i+seq_length] for i in ix])
    y = torch.stack([data[i+1:i+seq_length+1] for i in ix]) # Shifted by 1
    return x, y

Xb, Yb = get_batch()
print(f"Input shape: {Xb.shape}, Target shape: {Yb.shape}")

Input shape: torch.Size([4, 8]), Target shape: torch.Size([4, 8])


2. The Architecture: Adding the Causal Mask

We will use PyTorch's built-in nn.TransformerEncoderLayer, but we will pass it a mask. The mask is a lower-triangular matrix. It sets all "future" connections to $-\infty$. When passed through the Softmax function in the attention mechanism, $e^{-\infty}$ becomes $0$, effectively blinding the model to future tokens.

In [6]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers):
        super().__init__()
        # Token Embedding (Translates integers to dense vectors)
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        # Positional Embedding (Tells the model where the token is in the sequence)
        self.position_embedding = nn.Embedding(128, embed_dim)

        # Stack of Transformer Blocks
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=0.1,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Final Output Layer (Translates vectors back to vocabulary probabilities)
        self.lm_head = nn.Linear(embed_dim, vocab_size)

    def generate_square_subsequent_mask(self, sz):
        # Creates a matrix of -inf above the diagonal, and 0s below
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # Get embeddings
        tok_emb = self.token_embedding(idx) # (B, T, embed_dim)
        pos_emb = self.position_embedding(torch.arange(T, device=idx.device)) # (T, embed_dim)
        x = tok_emb + pos_emb # Inject positional information

        # Generate the causal mask so it can't look ahead
        mask = self.generate_square_subsequent_mask(T).to(idx.device)

        # Pass through Transformer
        x = self.transformer(x, mask=mask, is_causal=True)

        # Get raw logits for the next token predictions
        logits = self.lm_head(x) # (B, T, vocab_size)

        # Calculate Loss if targets are provided
        loss = None
        if targets is not None:
            # PyTorch CrossEntropy expects (Batch*Seq, Classes)
            B, T, C = logits.shape
            logits_reshaped = logits.view(B*T, C)
            targets_reshaped = targets.view(B*T)
            loss = F.cross_entropy(logits_reshaped, targets_reshaped)

        return logits, loss

3. The Optimization Loop

Now we bring in our optimizer (AdamW is standard for Transformers) and iterate over our data. The backpropagation step calculates the gradients, and the optimizer updates the parameters.

In [7]:
# Initialize Model and Optimizer
model = MiniGPT(vocab_size=vocab_size, embed_dim=64, num_heads=4, num_layers=2)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

epochs = 1000

print("Starting training loop...")
for iter in range(epochs):
    # 1. Get a batch of data
    xb, yb = get_batch()

    # 2. Forward Pass
    logits, loss = model(xb, targets=yb)

    # 3. Backward Pass
    optimizer.zero_grad(set_to_none=True) # Clear old gradients
    loss.backward()                       # Calculate new gradients

    # 4. Update Weights
    optimizer.step()

    if iter % 200 == 0:
        print(f"Epoch {iter} | Loss: {loss.item():.4f}")

print("Training complete!")

Starting training loop...
Epoch 0 | Loss: 3.0711
Epoch 200 | Loss: 0.6357
Epoch 400 | Loss: 0.2648
Epoch 600 | Loss: 0.2576
Epoch 800 | Loss: 0.2835
Training complete!


4. Inference (Generating Text)

Once trained, we can prompt the model. It feeds its own predictions back into itself (autoregression) to generate sequence text.

In [8]:
# Start with a single character prompt (e.g., "d")
context = torch.tensor((encode("d")), dtype=torch.long).unsqueeze(0)

# Generate 50 new characters
for _ in range(50):
    # Get predictions for the current context
    logits, _ = model(context)
    # Focus only on the last time step (the predicted next token)
    logits = logits[:, -1, :]
    # Get probabilities
    probs = F.softmax(logits, dim=-1)
    # Sample from the distribution
    idx_next = torch.multinomial(probs, num_samples=1)
    # Append to the context
    context = torch.cat((context, idx_next), dim=1)

print("\nGenerated Text:")
print(decode(context[0].tolist()))


Generated Text:
d transfowe nsfuasformeratrasf porfoweecininas trfo
